# $B^+\to K^+\pi^+\pi^-$ benchmark toy fit

This notebook keeps the full paper-inspired benchmark model while using the concise high-level workflow. The amplitude contains $K^*(892)^0$, the $(K\pi)_S$ LASS component, $\rho(770)^0$, $f_0(980)$ and a non-resonant term.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, DecayChannel, DecayModel, FitSession, LASS, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, generate_toy,
    plot_dalitz, plot_square_dalitz,
)

enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))

truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS":     (1.40, -0.60),
    "rho770":   (0.65, 0.10),
    "f0_980":   (-0.20, 1.00),
    "NR":       (-0.50, 0.10),
}
truth = {}

def coefficient(name, fixed=False):
    x, y = truth_xy[name]
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"] = x
    truth[f"{name}.y"] = y
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "Kstar892")) for name in truth_xy}
model = DecayModel(
    channel,
    [
        Resonance("Kstar892", (0,2), c["Kstar892"],
                  mass=0.8958, width=0.0474, spin=1,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("KpiS", (0,2), c["KpiS"],
                  lineshape=LASS(2.07, 3.32, 1.8),
                  mass=1.425, width=0.270, spin=0,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho770", (1,2), c["rho770"],
                  mass=0.7753, width=0.1491, spin=1,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("f0_980", (1,2), c["f0_980"],
                  lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0,
                  resonance_radius=4.0, parent_radius=4.0),
        NonResonant(c["NR"]),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=350,
    normalization_pair=(0,2),
)

print("Generated fit fractions:")
model.print_fit_fractions(truth, include_interference=True)


In [ ]:
data = generate_toy(
    model, 30_000, parameters=truth, seed=303, pool_size=250_000,
)
plot_dalitz(data, x="s13", y="s23", title="B+ benchmark toy")
plt.show()
plot_square_dalitz(
    data, mother_mass=channel.parent_mass, masses=channel.daughter_masses,
    pair=(0,2), title="B+ Square Dalitz"
)
plt.show()


In [ ]:
session = FitSession(model, data)
rng = np.random.default_rng(314159)
start = {
    p.name: truth[p.name] + rng.normal(0.0, 0.12)
    for p in session.parameters if not p.fixed
}
result = session.fit(start, simplex=True, ncall=50_000)

session.report(result)
print("\nFitted fractions with interference terms:")
session.print_fit_fractions(result, include_interference=True)
session.plot_projection(result, "s13")
plt.show()
session.plot_projection(result, "s23")
plt.show()
